# TF-IDF Retrieval

This notebook implements the retrieval component of the retrieval-augmented generation (RAG) pipeline.

## Purpose

TF-IDF is used as a sparse retrieval baseline to retrieve the course-material chunks that are most relevant to a student question. The retrieved chunks can later be passed to an LLM or a sequence-to-sequence model to generate an answer.

## Pipeline

1. Load the preprocessed course-material chunks created in Notebook 03.
2. Build a TF-IDF index from the course-material chunks.
3. Retrieve the top-$k$ chunks for each question using cosine similarity.
4. Evaluate retrieval performance against the annotated source pages.
5. Use the retrieved chunks as context for the LLM and sequence-to-sequence experiments.

- **Train:** retrieved contexts are used as input to the LLM and sequence-to-sequence experiments.
- **Validation:** retrieved contexts are used to tune and evaluate the retrieval and generation pipeline.
- **Test:** retrieved contexts are used for the final, held-out evaluation.

## Data flow

- **Input documents:** `data/processed/chunks_preprocessed.jsonl`
- **Questions:** `data/splits/train.jsonl`, `data/splits/validation.jsonl`, and `data/splits/test.jsonl`
- **Retriever:** TF-IDF with cosine similarity
- **Output:** ranked chunks, retrieved page references, and retrieval metrics

The TF-IDF retriever is fitted on the course-material chunks. The same fitted retriever is then used to retrieve relevant chunks for the train, validation, and test questions.

## RAG Retrieval Pipeline

```text
Notebook 03
Preprocessing and splitting
        |
        v
Preprocessed course chunks
        |
        v
Notebook 04
TF-IDF retriever
        |
        +------------------+
        |                  |
        v                  v
Train questions      Validation questions
retrieved contexts   tune and evaluate
        |
        v
Answer generator
        |
        +------------------+
        |                  |
        v                  v
      LLM              Seq2Seq
        |                  |
        +--------+---------+
                 v
              Answers

Test questions
        |
        v
Final retrieval evaluation
        |
        v
Final LLM and Seq2Seq evaluation
```

In [36]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
TOP_K_VALUES = (1, 3, 5, 10)

In [38]:
PROJECT_ROOT = Path.cwd()

PREPROCESSING_NOTEBOOK_PATH = (
    PROJECT_ROOT
    / "03_proccessingSplitting.ipynb"
)

CHUNKS_PREPROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)

TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "train.jsonl"
)

VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "validation.jsonl"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "test.jsonl"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PREPROCESSING_NOTEBOOK_PATH.exists()
assert CHUNKS_PREPROCESSED_PATH.exists()
assert TRAIN_PATH.exists()
assert VALIDATION_PATH.exists()
assert TEST_PATH.exists()

print("Notebook 03:", PREPROCESSING_NOTEBOOK_PATH)
print("Chunkovi:", CHUNKS_PREPROCESSED_PATH)
print("Trening skup:", TRAIN_PATH)
print("Validacioni skup:", VALIDATION_PATH)
print("Test skup:", TEST_PATH)

Notebook 03: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/03_proccessingSplitting.ipynb
Chunkovi: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/processed/chunks_preprocessed.jsonl
Trening skup: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/train.jsonl
Validacioni skup: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/validation.jsonl
Test skup: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/test.jsonl


In [39]:
with PREPROCESSING_NOTEBOOK_PATH.open("r", encoding="utf-8") as file:
    notebook_03 = json.load(file)

load_jsonl_source = next(
    cell["source"]
    for cell in notebook_03["cells"]
    if cell.get("cell_type") == "code"
    and any(
        line.startswith("def load_jsonl")
        for line in cell.get("source", [])
    )
)

exec("".join(load_jsonl_source), globals())
print("Funkcija load_jsonl je učitana iz notebooka 03.")

Funkcija load_jsonl je učitana iz notebooka 03.


In [40]:
chunks = load_jsonl(CHUNKS_PREPROCESSED_PATH)

train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

print(f"Broj chunkova: {len(chunks)}")
print(f"Broj pitanja u trening skupu: {len(train_data)}")
print(f"Broj pitanja u validacionom skupu: {len(validation_data)}")
print(f"Broj pitanja u test skupu: {len(test_data)}")

Broj chunkova: 344
Broj pitanja u trening skupu: 100
Broj pitanja u validacionom skupu: 21
Broj pitanja u test skupu: 22


In [41]:
assert chunks

assert all(
    isinstance(chunk.get("lexical_text"), str)
    and chunk["lexical_text"].strip()
    for chunk in chunks
)

for split_name, data in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data
}.items():

    assert data, f"{split_name} split is empty"

    assert all(
        isinstance(example.get("lexical_question"), str)
        and example["lexical_question"].strip()
        for example in data
    ), f"Missing lexical_question in {split_name}"

In [42]:
chunks_df = pd.DataFrame(chunks)

train_df = pd.DataFrame(train_data)
validation_df = pd.DataFrame(validation_data)
test_df = pd.DataFrame(test_data)

chunks_df[
    ["chunk_id", "pdf_page_start", "pdf_page_end", "lexical_text"]
].head(3)

,chunk_id,pdf_page_start,pdf_page_end,lexical_text
0,chunk_0001,17,17,pregled 1 1 upravljanje kvalitetom softvera 4 ...
1,chunk_0002,17,17,avstvene zaštite takođe softver ima ključnu ul...
2,chunk_0003,17,18,upravljanjem procesima njegove izrade u proces...


In [43]:
chunk_ids = chunks_df["chunk_id"].tolist()

chunk_texts = chunks_df["lexical_text"].tolist()

print(f"Broj chunkova: {len(chunk_texts)}")
print(f"Broj ID-ova chunkova: {len(chunk_ids)}")

Broj chunkova: 344
Broj ID-ova chunkova: 344


## TF-IDF Vectorization

The document chunks are converted into TF-IDF vectors using unigrams and bigrams. Serbian stop words are removed so that common function words contribute less to retrieval scores.

In [44]:
serbian_stop_words = {
    "i", "u", "na", "je", "su", "se", "za", "od", "do",
    "sa", "kao", "koji", "koja", "koje", "što", "da",
    "ili", "a", "ali", "po", "iz", "uz", "o", "pri",
    "bi", "biti", "ima", "imaju"
}

vectorizer = TfidfVectorizer(
    lowercase=False,
    stop_words=list(serbian_stop_words),
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

In [45]:
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("Oblik matrice TF-IDF :", tfidf_matrix.shape)
print("Velicina rečnika:", len(vectorizer.vocabulary_))
print(f"Broj nenultih vrednosti: {tfidf_matrix.nnz}")

Oblik matrice TF-IDF : (344, 31634)
Velicina rečnika: 31634
Broj nenultih vrednosti: 64672


## Inspect TF-IDF Terms

This section shows which terms receive the highest TF-IDF weights in one document chunk. These weights provide an interpretable view of the words and bigrams that characterize that chunk.

In [46]:
chunk_index = 0

feature_names = np.array(
    vectorizer.get_feature_names_out()
)

vector = tfidf_matrix[chunk_index]

scores = vector.toarray().flatten()

top_indices = scores.argsort()[::-1][:15]

tfidf_terms_df = pd.DataFrame({
    "term": feature_names[top_indices],
    "tfidf": scores[top_indices]
})

tfidf_terms_df

,term,tfidf
0,industrija,0.144224
1,standardi,0.108954
2,važni,0.108954
3,razvija,0.100331
4,kvaliteta softvera,0.096414
5,atributi kvaliteta,0.086952
6,atributi,0.085954
7,kvaliteta,0.085792
8,tokom poslednjih,0.085181
9,jednu najdinamičnijih,0.085181


In [47]:
example = validation_data[0]

query = example["lexical_question"]

query_vector = vectorizer.transform([query])

query_scores = query_vector.toarray().flatten()

nonzero = np.where(query_scores > 0)[0]

query_terms_df = pd.DataFrame({
    "term": feature_names[nonzero],
    "tfidf": query_scores[nonzero]
}).sort_values(
    "tfidf",
    ascending=False
)

query_terms_df

,term,tfidf
4,šta,0.625603
3,služi,0.540498
0,cachegrind,0.479585
2,koristi,0.207943
1,kako,0.207943


In [48]:
import altair as alt

alt.Chart(query_terms_df).mark_bar().encode(
    x=alt.X(
        "tfidf:Q",
        title="TF-IDF težina"
    ),
    y=alt.Y(
        "term:N",
        sort="-x",
        title="Termin"
    ),
    tooltip=["term", "tfidf"]
).properties(
    title="TF-IDF reprezentacija pitanja",
    width=500,
    height=300
)

alt.Chart(...)

## Single Query Retrieval

A single lexical question is transformed into a TF-IDF vector and compared with every document chunk. The highest-scoring chunks are returned as the query's ranked retrieval results.

In [49]:
def retrieve_chunks(
    lexical_question: str,
    top_k: int = 5
) -> pd.DataFrame:

    question_vector = vectorizer.transform(
        [lexical_question]
    )

    if question_vector.nnz == 0:
        return pd.DataFrame(columns=[
            "rank",
            "chunk_id",
            "score",
            "pdf_page_start",
            "pdf_page_end",
            "processed_text",    # bolja čitljivost
        ])
    
    similarities = cosine_similarity(
        question_vector,
        tfidf_matrix
    ).flatten()

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        chunk = chunks[index]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "score": float(similarities[index]),
            "pdf_page_start": chunk["pdf_page_start"],
            "pdf_page_end": chunk["pdf_page_end"],
            "processed_text": chunk["processed_text"]   # bolja čitljivost
        })

    return pd.DataFrame(results)

In [50]:
def get_relevant_chunk_ids(gold_pages):
    gold_pages = set(gold_pages)

    relevant_ids = set()

    for chunk in chunks:
        chunk_pages = set(
            range(
                int(chunk["pdf_page_start"]),
                int(chunk["pdf_page_end"]) + 1
            )
        )

        if chunk_pages & gold_pages:
            relevant_ids.add(chunk["chunk_id"])

    return relevant_ids

In [51]:
example = validation_data[1]

print("Original question:")
print(example["question"])

print("\nProcessed question:")
print(example["lexical_question"])

print("\nGold source pages:")
print(example["source_pages"])

Original question:
Šta je instrumentaciono profajliranje?

Processed question:
šta je instrumentaciono profajliranje

Gold source pages:
[188, 189]


In [52]:
def overlaps_gold_pages(
    row,
    source_pages
):
    return any(
        row["pdf_page_start"] <= page <= row["pdf_page_end"]
        for page in source_pages
    )

In [53]:
retrieved = retrieve_chunks(
    example["lexical_question"],
    top_k=10
)

retrieved["relevant"] = retrieved.apply(
    lambda row: overlaps_gold_pages(
        row,
        example["source_pages"]
    ),
    axis=1
)

retrieved[
    [
        "rank",
        "score",
        "pdf_page_start",
        "pdf_page_end",
        "relevant",
        "processed_text"
    ]
]

,rank,score,pdf_page_start,pdf_page_end,relevant,processed_text
0,1,0.124642,185,185,False,[Profajleri] uhvaćena uzorkovanjem. Ovaj prist...
1,2,0.096824,187,188,True,"[Profajleri] (""*""); 39 } 40 myWriter.write(Sys..."
2,3,0.054706,174,175,False,[Profajliranje i dinamičko detektovanje grešak...
3,4,0.049212,188,188,True,[Profajliranje i dinamičko detektovanje grešak...
4,5,0.042742,174,174,False,[Profajliranje i dinamičko detektovanje grešak...
5,6,0.037978,180,181,False,[Profajliranje i dinamičko detektovanje grešak...
6,7,0.037430,181,181,False,[Profajleri] i otključavanja katanaca prilikom...
7,8,0.037003,184,184,False,[Profajliranje i dinamičko detektovanje grešak...
8,9,0.031218,180,180,False,[Profajliranje i dinamičko detektovanje grešak...
9,10,0.028516,184,185,False,[Profajliranje i dinamičko detektovanje grešak...


## Visualize Similarities

Cosine similarity measures how closely the question matches each chunk in the TF-IDF space. The top 20 scores make the retrieval ranking visible and show how sharply the best matches stand out.

In [54]:
example = validation_data[0]

question_vector = vectorizer.transform(
    [example["lexical_question"]]
)

similarities = cosine_similarity(
    question_vector,
    tfidf_matrix
).flatten()

similarity_df = pd.DataFrame({
    "chunk_index": range(len(similarities)),
    "similarity": similarities
}).sort_values(
    "similarity",
    ascending=False
)

top_similarity_df = similarity_df.head(20)

top_similarity_df

,chunk_index,similarity
314,314,0.104376
303,303,0.058202
310,310,0.053463
143,143,0.048506
92,92,0.047647
276,276,0.044445
311,311,0.044188
327,327,0.042049
94,94,0.041330
326,326,0.040257


In [55]:
alt.Chart(top_similarity_df).mark_bar().encode(
    x=alt.X(
        "similarity:Q",
        title="Cosine similarity"
    ),
    y=alt.Y(
        "chunk_index:O",
        sort="-x",
        title="Chunk"
    ),
    tooltip=["chunk_index", "similarity"]
).properties(
    title="TF-IDF similarity pitanja sa top chunkovima",
    width=500,
    height=400
)

alt.Chart(...)

## Retrieval Evaluation

The retriever is evaluated by comparing the retrieved chunk IDs with the chunks whose page ranges overlap the annotated source pages. This produces per-query relevance values that are aggregated across the validation set.

In [56]:
def evaluate_example(example, top_k=10):

    retrieved = retrieve_chunks(
        example["lexical_question"],
        top_k=top_k
    )

    relevant_ids = get_relevant_chunk_ids(
        example["source_pages"]
    )

    retrieved_ids = retrieved["chunk_id"].tolist()

    relevances = [
        1 if chunk_id in relevant_ids else 0
        for chunk_id in retrieved_ids
    ]

    # Hit@k - Pogodak u prvih k rezultata
    hit = int(any(relevances))

    # Precision@k - Preciznost u prvih k rezultata
    precision = sum(relevances) / top_k

    # True Recall@k - Stvarni odziv u prvih k rezultata
    recall = (
        sum(relevances) / len(relevant_ids)
        if relevant_ids
        else 0
    )

    # Recipročni rang prvog relevantnog rezultata
    reciprocal_rank = 0.0

    for rank, relevant in enumerate(relevances, start=1):
        if relevant:
            reciprocal_rank = 1.0 / rank
            break

    return {
        "hit": hit,
        "precision": precision,
        "recall": recall,
        "reciprocal_rank": reciprocal_rank
    }

In [57]:
def evaluate_retriever(data, top_k=10):

    results = [
        evaluate_example(example, top_k)
        for example in data
    ]

    return {
        f"Hit@{top_k}":
            np.mean([r["hit"] for r in results]),

        f"Precision@{top_k}":
            np.mean([r["precision"] for r in results]),

        f"Recall@{top_k}":
            np.mean([r["recall"] for r in results]),

        f"MRR@{top_k}":
            np.mean([r["reciprocal_rank"] for r in results])
    }

## Hit@k / Precision@k / Recall@k / MRR

These metrics describe complementary aspects of retrieval quality: whether a relevant chunk is found, how many retrieved chunks are relevant, how much of the relevant material is covered, and how high the first relevant result appears. They are reported for several values of $k$ to show how retrieval quality changes as more chunks are considered.

In [58]:
evaluate_retriever(
    validation_data,
    top_k=5
)

{'Hit@5': np.float64(0.47619047619047616),
 'Precision@5': np.float64(0.20952380952380953),
 'Recall@5': np.float64(0.22828798185941043),
 'MRR@5': np.float64(0.3547619047619048)}

In [59]:
metrics_rows = []

for k in TOP_K_VALUES:

    metrics = evaluate_retriever(
        validation_data,
        top_k=k
    )

    metrics_rows.append({
        "k": k,
        "Hit": metrics[f"Hit@{k}"],
        "Precision": metrics[f"Precision@{k}"],
        "Recall": metrics[f"Recall@{k}"],
        "MRR": metrics[f"MRR@{k}"]
    })

metrics_df = pd.DataFrame(metrics_rows)

metrics_df

,k,Hit,Precision,Recall,MRR
0,1,0.285714,0.285714,0.059921,0.285714
1,3,0.380952,0.206349,0.137302,0.333333
2,5,0.476190,0.209524,0.228288,0.354762
3,10,0.523810,0.157143,0.312132,0.362698


In [60]:
import altair as alt

alt.Chart(tfidf_terms_df).mark_bar().encode(
    x=alt.X(
        "tfidf:Q",
        title="TF-IDF težina"
    ),
    y=alt.Y(
        "term:N",
        sort="-x",
        title="Termin"
    ),
    tooltip=["term", "tfidf"]
).properties(
    title="Najvažniji TF-IDF termini u chunku",
    width=500,
    height=350
)

alt.Chart(...)

In [61]:
import altair as alt

metrics_plot_df = metrics_df.melt(
    id_vars="k",
    value_vars=["Hit", "Precision", "Recall", "MRR"],
    var_name="metric",
    value_name="score"
)

chart = alt.Chart(metrics_plot_df).mark_line(point=True).encode(
    x=alt.X("k:O", title="Top-k"),
    y=alt.Y(
        "score:Q",
        title="Vrednost metrike",
        scale=alt.Scale(domain=[0, 1])
    ),
    color=alt.Color("metric:N", title="Metrika"),
    tooltip=["k", "metric", "score"]
).properties(
    title="Performanse TF-IDF pretraživača",
    width=500,
    height=300
)

chart

alt.Chart(...)

## Saving TF-IDF Artifacts

The fitted vectorizer, TF-IDF matrix, chunk metadata, validation metrics, and top-k retrieval results are saved for reuse. These artifacts are providing the shared retrieval input for the Qwen and mT5 experiments.

In [62]:
import json

import joblib
from scipy import sparse

ARTIFACTS_DIR = RESULTS_DIR / "tfidf"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    vectorizer,
    ARTIFACTS_DIR / "tfidf_vectorizer.joblib"
)

sparse.save_npz(
    ARTIFACTS_DIR / "tfidf_matrix.npz",
    tfidf_matrix
)

chunks_export = chunks_df[
    [
        "chunk_id",
        "text",
        "processed_text",
        "lexical_text",
        "pdf_page_start",
        "pdf_page_end",
        "printed_page_start",
        "printed_page_end",
        "section_ref",
        "heading",
    ]
]

chunks_export.to_json(
    ARTIFACTS_DIR / "tfidf_chunks.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

metadata = {
    "vectorizer_input": "lexical_text",
    "retrieval_question_input": "lexical_question",
    "generator_context_input": "processed_text",
    "top_k": max(TOP_K_VALUES),
    "n_chunks": len(chunks),
    "n_features": len(vectorizer.get_feature_names_out()),
}

with (ARTIFACTS_DIR / "tfidf_metadata.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

metrics_df.to_csv(
    ARTIFACTS_DIR / "validation_metrics.csv",
    index=False
)

print("TF-IDF artefakti su sačuvani u:", ARTIFACTS_DIR)

TF-IDF artefakti su sačuvani u: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/retrieval/tfidf


In [63]:
def export_retrieval_results(data, split_name, top_k=10):
    export_path = RESULTS_DIR / f"tfidf_{split_name}_top{top_k}.jsonl"

    with export_path.open("w", encoding="utf-8") as file:
        for example in data:
            retrieved = retrieve_chunks(
                example["lexical_question"],
                top_k=top_k
            )

            record = {
                "question_id": example["id"],
                "question": example["question"],
                "processed_question": example["processed_question"],
                "lexical_question": example["lexical_question"],
                "answer": example["answer"],
                "source_pages": example["source_pages"],
                "retrieved_chunks": retrieved.to_dict(orient="records"),
            }

            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    print(f"Sačuvani retrieval rezultati za {split_name}:", export_path)


for split_name, data in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
}.items():
    export_retrieval_results(
        data,
        split_name,
        top_k=max(TOP_K_VALUES)
    )

Sačuvani retrieval rezultati za train: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/retrieval/tfidf_train_top10.jsonl
Sačuvani retrieval rezultati za validation: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/retrieval/tfidf_validation_top10.jsonl
Sačuvani retrieval rezultati za test: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/retrieval/tfidf_test_top10.jsonl
